# Module 4: Offline Validation and Ablation Study

This notebook validates Module 3 outputs against README Module 4 requirements.

Implementation rule alignment: all heavy evaluation logic is imported from `src/module4_validation.py` and `src/cold_start.py`.

Covered here:
- temporal holdout split (Weeks 81-102 when available, Day 601-711 fallback),
- ablation variants for the utility function,
- incremental precision@5 and margin lift diagnostics,
- margin lift vs popularity baseline,
- cold-start recommendation demo from demographic priors.

In [4]:
from pathlib import Path
import importlib.util
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_RAW = ROOT / 'data' / '01_raw'
DATA_PROCESSED = ROOT / 'data' / '02_processed'
REPORTS = ROOT / 'reports'
FIGURES = REPORTS / 'figures'

def read_csv_or_none(path: Path):
    if not path.exists():
        return None
    frame = pd.read_csv(path)
    if list(frame.columns) == ['version https://git-lfs.github.com/spec/v1']:
        return None
    return frame

def package_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

print('Project root:', ROOT)
print('plotly available:', package_available('plotly'))

Project root: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey
plotly available: True


In [5]:
from src.data_loader import load_or_build_master_transactions
from src.cold_start import build_demographic_priors, recommend_for_new_user
from src.module4_validation import (
    build_ablation_weight_templates,
    build_temporal_holdout,
    run_ablation,
    build_variant_topk,
    make_variant_weights,
    evaluate_incremental_precision,
    build_popularity_baseline_topk,
    attach_margin_to_topk,
)

campaign_table = pd.read_csv(DATA_RAW / 'campaign_table.csv')
campaign_desc = pd.read_csv(DATA_RAW / 'campaign_desc.csv')
causal_data = pd.read_csv(DATA_RAW / 'causal_data.csv')
hh_demographic = pd.read_csv(DATA_RAW / 'hh_demographic.csv')
product_catalog = pd.read_csv(DATA_RAW / 'product.csv', usecols=['PRODUCT_ID', 'COMMODITY_DESC', 'BRAND'])

# Avoid cached parquet schema issues in master transaction artifacts.
master_tx_dir = DATA_PROCESSED
all_path = master_tx_dir / 'master_transactions_all.parquet'
organic_path = master_tx_dir / 'master_transactions_organic_only.parquet'
if all_path.exists() or organic_path.exists():
    master_tx_dir = master_tx_dir / 'notebook_tmp'
    master_tx_dir.mkdir(parents=True, exist_ok=True)

tx_all, tx_organic = load_or_build_master_transactions(DATA_RAW, master_tx_dir)
commodity_margin = read_csv_or_none(DATA_PROCESSED / 'commodity_margin.csv')
scored_candidates = read_csv_or_none(DATA_PROCESSED / 'candidate_set_module3_scored.csv')
top5 = read_csv_or_none(DATA_PROCESSED / 'top5_recommendations_module3.csv')

if commodity_margin is None:
    raise FileNotFoundError('Module 3 commodity_margin.csv is required before running validation.')
if scored_candidates is None:
    raise FileNotFoundError('Module 3 candidate_set_module3_scored.csv is required before running validation.')
if top5 is None:
    raise FileNotFoundError('Module 3 top5_recommendations_module3.csv is required before running validation.')

print('tx_all rows:', len(tx_all))
print('tx_organic rows:', len(tx_organic))
print('scored_candidates rows:', len(scored_candidates))
print('top5 rows:', len(top5))
print('hh_demographic rows:', len(hh_demographic))
display(scored_candidates.head(5))

tx_all rows: 2595732
tx_organic rows: 96240
scored_candidates rows: 23544
top5 rows: 2500
hh_demographic rows: 801


,household_key,COMMODITY_DESC,seed_items,relevance_als,relevance_mba,Relevance,source,source_detail,snapshot_week,was_recently_purchased,...,Normalized_Margin,deal_sensitivity,is_active_campaign_period,item_is_promoted,Context,Utility,Relevance_Contribution,Uplift_Contribution,Margin_Contribution,Context_Contribution
0,1,POPCORN,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.947021,0.000000,0.947021,als,ALS,102,False,...,1.0,0.813953,False,True,0.5,0.900901,0.378808,0.247093,0.2,0.075
1,1,VEGETABLES - ALL OTHERS,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.000000,1.000000,1.000000,mba,MBA,102,False,...,1.0,0.813953,False,True,0.5,0.898837,0.400000,0.223837,0.2,0.075
2,1,BEEF,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.000000,0.796527,0.796527,mba,MBA,102,False,...,1.0,0.813953,False,True,0.5,0.837797,0.318611,0.244186,0.2,0.075
3,1,BLEACH,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.721154,0.000000,0.721154,als,ALS,102,False,...,1.0,0.813953,False,True,0.5,0.798927,0.288462,0.235465,0.2,0.075
4,1,CHRISTMAS SEASONAL,BAKED BREAD/BUNS/ROLLS | CANDY - PACKAGED | CA...,0.523417,0.000000,0.523417,als,ALS,102,False,...,1.0,0.813953,False,True,0.5,0.728553,0.209367,0.244186,0.2,0.075


In [6]:
score_columns = ['Relevance', 'Uplift', 'Normalized_Margin', 'Context']
required_scored_columns = ['household_key', 'COMMODITY_DESC', *score_columns]

missing_scored_columns = [column for column in required_scored_columns if column not in scored_candidates.columns]
if missing_scored_columns:
    raise ValueError(f'Module 3 scored output is missing required columns: {missing_scored_columns}')

scored_candidates = scored_candidates.copy()
scored_candidates['household_key'] = pd.to_numeric(scored_candidates['household_key'], errors='coerce').astype('Int64')
scored_candidates = scored_candidates.dropna(subset=['household_key', 'COMMODITY_DESC'])
scored_candidates['household_key'] = scored_candidates['household_key'].astype(int)

split = build_temporal_holdout(tx_all, holdout_weeks=range(81, 103), day_split=(600, 711))
ablation_weight_templates = build_ablation_weight_templates()

print('holdout weeks:', split.selected_weeks[:5], '...', split.selected_weeks[-5:] if len(split.selected_weeks) > 5 else split.selected_weeks)
print('training rows:', len(split.train_history))
print('test rows:', len(split.test_history))
print('eligible households:', len(split.eligible_households))
display(ablation_weight_templates)

holdout weeks: [81, 82, 83, 84, 85] ... [98, 99, 100, 101, 102]
training rows: 1968369
test rows: 627363
eligible households: 2408


,Variant,relevance,uplift,margin,context
0,Variant 0 - Relevance only,1.00,0.00,0.0,0.00
1,Variant 1 - Relevance + Uplift,0.75,0.25,0.0,0.00
2,Variant 2 - Relevance + Uplift + Margin,0.60,0.20,0.2,0.00
3,Variant 3 - Full Chimera,0.40,0.25,0.2,0.15


In [7]:
ablation_summary, variant_user_metrics_long, variant_outputs = run_ablation(
    scored_candidates=scored_candidates,
    split=split,
    weight_templates=ablation_weight_templates,
    margin_lookup=commodity_margin,
    top_k=5,
)

display(ablation_summary)

,Variant,Incremental_Precision@5,Average_Recommended_Margin,Average_Incremental_Hits,Average_Targets,Precision_Lift_vs_Baseline,Margin_Lift_vs_Baseline,Margin_Lift_vs_Popularity
0,Variant 0 - Relevance only,0.064301,0.886848,0.321503,11.609603,0.000000,0.000000,0.007135
1,Variant 1 - Relevance + Uplift,0.065971,0.854697,0.329854,11.609603,0.025974,-0.036252,-0.029376
2,Variant 2 - Relevance + Uplift + Margin,0.067641,0.963674,0.338205,11.609603,0.051948,0.086629,0.094382
3,Variant 3 - Full Chimera,0.069311,0.985386,0.346555,11.609603,0.077922,0.111111,0.119039


In [8]:
FIGURES.mkdir(parents=True, exist_ok=True)

summary_long = ablation_summary.melt(
    id_vars='Variant',
    value_vars=['Incremental_Precision@5', 'Average_Recommended_Margin'],
    var_name='Metric',
    value_name='Value',
)

fig_ablation = px.bar(
    summary_long,
    x='Variant',
    y='Value',
    color='Metric',
    barmode='group',
    title='Module 4 Ablation Impact: Incremental Precision@5 vs Average Recommended Margin',
)
fig_ablation.update_layout(height=460, xaxis_title='', yaxis_title='Score')
fig_ablation.show()

fig_precision_box = px.box(
    variant_user_metrics_long,
    x='Variant',
    y='incremental_precision_at_5',
    points='all',
    title='Distribution of Household Incremental Precision@5 by Variant',
)
fig_precision_box.update_layout(height=460, xaxis_title='', yaxis_title='Incremental Precision@5')
fig_precision_box.show()

fig_lift = px.bar(
    ablation_summary,
    x='Variant',
    y=['Precision_Lift_vs_Baseline', 'Margin_Lift_vs_Baseline', 'Margin_Lift_vs_Popularity'],
    barmode='group',
    title='Relative Lift Diagnostics vs Baseline and Popularity',
)
fig_lift.update_layout(height=460, xaxis_title='', yaxis_title='Relative Lift')
fig_lift.show()

heatmap_data = variant_user_metrics_long.pivot_table(
    index='household_key',
    columns='Variant',
    values='incremental_precision_at_5',
    aggfunc='mean',
).fillna(0.0)

if not heatmap_data.empty:
    fig_heatmap = go.Figure(
        data=go.Heatmap(
            z=heatmap_data.values,
            x=heatmap_data.columns,
            y=heatmap_data.index,
            colorscale='Viridis',
            colorbar=dict(title='P@5'),
        )
    )
    fig_heatmap.update_layout(
        title='Household-Level Incremental Precision Heatmap by Variant',
        xaxis_title='Variant',
        yaxis_title='Household',
        height=520,
    )
    fig_heatmap.show()

fig_ablation.write_html(FIGURES / 'ablation_lift_chart.html')
fig_precision_box.write_html(FIGURES / 'ablation_precision_distribution.html')
fig_lift.write_html(FIGURES / 'ablation_relative_lift.html')
if not heatmap_data.empty:
    fig_heatmap.write_html(FIGURES / 'ablation_household_heatmap.html')

print('saved figures:', FIGURES / 'ablation_lift_chart.html')
print('saved figures:', FIGURES / 'ablation_precision_distribution.html')
print('saved figures:', FIGURES / 'ablation_relative_lift.html')
if not heatmap_data.empty:
    print('saved figures:', FIGURES / 'ablation_household_heatmap.html')

saved figures: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\reports\figures\ablation_lift_chart.html
saved figures: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\reports\figures\ablation_precision_distribution.html
saved figures: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\reports\figures\ablation_relative_lift.html
saved figures: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\reports\figures\ablation_household_heatmap.html


In [9]:
train_popularity = (
    split.train_history.groupby('COMMODITY_DESC')
    .agg(purchase_count=('BASKET_ID', 'nunique'), revenue=('Revenue_Retailer', 'sum'))
    .reset_index()
    .sort_values(['purchase_count', 'revenue'], ascending=False)
    .rename(columns={'purchase_count': 'popularity_score'})
    [['COMMODITY_DESC', 'popularity_score']]
 )

if not train_popularity.empty:
    train_popularity['popularity_score'] = train_popularity['popularity_score'] / train_popularity['popularity_score'].max()

demographic_priors = build_demographic_priors(hh_demographic, train_popularity)

if demographic_priors.empty:
    print('No demographic priors could be built from the available columns.')
else:
    sample_household = int(demographic_priors['household_key'].iloc[0])
    sample_profile = hh_demographic[hh_demographic['household_key'] == sample_household].head(1)
    fallback_items = demographic_priors[demographic_priors['household_key'] == sample_household][['COMMODITY_DESC', 'prior_score']].copy()
    cold_start_recs = recommend_for_new_user(sample_profile, top_k=10, fallback_items=fallback_items)
    print('Sample household:', sample_household)
    display(sample_profile)
    display(cold_start_recs)

ablation_summary_path = DATA_PROCESSED / 'module4_ablation_summary.csv'
ablation_summary.to_csv(ablation_summary_path, index=False)
print('saved summary:', ablation_summary_path)
print('cold-start priors rows:', len(demographic_priors))

Sample household: 1


,classification_1,classification_2,classification_3,HOMEOWNER_DESC,classification_5,classification_4,KID_CATEGORY_DESC,household_key
0,Age Group6,X,Level4,Homeowner,Group5,2,None/Unknown,1


,COMMODITY_DESC,prior_score
0,SOFT DRINKS,1.000000
1,FLUID MILK PRODUCTS,0.943639
2,BAKED BREAD/BUNS/ROLLS,0.831240
3,CHEESE,0.633885
4,BAG SNACKS,0.576695
5,BEEF,0.508766
6,TROPICAL FRUIT,0.446026
7,COUPON/MISC ITEMS,0.381628
8,EGGS,0.376511
9,REFRGRATD JUICES/DRNKS,0.365250


saved summary: C:\Users\dell\Downloads\New folder (9)\Data_Driven_Marketing_complete-journey\data\02_processed\module4_ablation_summary.csv
cold-start priors rows: 245106


## Module 4 Completion Check

The final cell validates the exported metrics, confirms the ablation study ran end to end, and highlights the best-performing variant under the current holdout split.

In [10]:
required_summary_cols = ['Variant', 'Incremental_Precision@5', 'Average_Recommended_Margin', 'Margin_Lift_vs_Popularity']
missing_summary_cols = [column for column in required_summary_cols if column not in ablation_summary.columns]

if missing_summary_cols:
    raise ValueError(f'Module 4 summary is incomplete: {missing_summary_cols}')

if ablation_summary.empty:
    raise ValueError('Module 4 ablation summary is empty.')

if ablation_summary[['Incremental_Precision@5', 'Average_Recommended_Margin']].isna().any().any():
    raise ValueError('Module 4 ablation summary contains missing values.')

best_variant = ablation_summary.sort_values(['Incremental_Precision@5', 'Average_Recommended_Margin'], ascending=[False, False]).iloc[0]

print('Module 4 completed successfully.')
print('Best variant under current holdout split:', best_variant['Variant'])
print('Incremental Precision@5:', float(best_variant['Incremental_Precision@5']))
print('Average Recommended Margin:', float(best_variant['Average_Recommended_Margin']))
print('Margin Lift vs Popularity:', float(best_variant['Margin_Lift_vs_Popularity']))
display(ablation_summary.sort_values('Incremental_Precision@5', ascending=False))

Module 4 completed successfully.
Best variant under current holdout split: Variant 3 - Full Chimera
Incremental Precision@5: 0.06931106471816284
Average Recommended Margin: 0.9853862212943633
Margin Lift vs Popularity: 0.11903887043804318


,Variant,Incremental_Precision@5,Average_Recommended_Margin,Average_Incremental_Hits,Average_Targets,Precision_Lift_vs_Baseline,Margin_Lift_vs_Baseline,Margin_Lift_vs_Popularity
3,Variant 3 - Full Chimera,0.069311,0.985386,0.346555,11.609603,0.077922,0.111111,0.119039
2,Variant 2 - Relevance + Uplift + Margin,0.067641,0.963674,0.338205,11.609603,0.051948,0.086629,0.094382
1,Variant 1 - Relevance + Uplift,0.065971,0.854697,0.329854,11.609603,0.025974,-0.036252,-0.029376
0,Variant 0 - Relevance only,0.064301,0.886848,0.321503,11.609603,0.000000,0.000000,0.007135
